# U'rWay Keyword Model Training

This notebook trains a text-to-keywords model on the custom `Urway.csv` dataset.

## Pipeline
1. Load only the `Description` and `Important Technology Keywords` columns.
2. Split the data into 80% train and 20% test.
3. Fine-tune a stronger instruction-tuned seq2seq model.
4. Evaluate on the held-out test split.
5. Try your own description in the final checker cell.

In [1]:
import os
import re
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

from project_config import (
    BASE_DIR,
    DATA_PATH,
    EPOCHS,
    LEARNING_RATE,
    MAX_INPUT_LEN,
    MAX_TARGET_LEN,
    MODEL_NAME,
    BATCH_SIZE,
)

MODEL_OUTPUT_DIR = os.path.join(BASE_DIR, "models", "u_rway_keyword_model")
CHECKPOINT_DIR = os.path.join(MODEL_OUTPUT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"✅ Dataset: {DATA_PATH}")
print(f"✅ Model: {MODEL_NAME}")
print(f"✅ Output dir: {MODEL_OUTPUT_DIR}")

c:\Users\HIMPADMA\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Dataset: c:\Users\HIMPADMA\Desktop\Dokumen\University Work\U'rWay\U'rWay Text Summarization\Urway.csv
✅ Model: google/flan-t5-small
✅ Output dir: c:\Users\HIMPADMA\Desktop\Dokumen\University Work\U'rWay\U'rWay Text Summarization\models\u_rway_keyword_model


In [2]:
# Load the custom dataset and keep only the two required columns
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Could not find dataset at {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
required_columns = {"Description", "Important Technology Keywords"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

df = df.loc[:, ["Description", "Important Technology Keywords"]].copy()
df = df.dropna(subset=["Description", "Important Technology Keywords"]).copy()
df["Description"] = df["Description"].astype(str).str.strip()
df["Important Technology Keywords"] = df["Important Technology Keywords"].astype(str).str.strip()
df = df[(df["Description"] != "") & (df["Important Technology Keywords"] != "")].reset_index(drop=True)

def normalize_keywords(text: str) -> str:
    parts = [p.strip() for p in str(text).split(",") if p.strip()]
    return ", ".join(parts)

df["Important Technology Keywords"] = df["Important Technology Keywords"].apply(normalize_keywords)

print(f"📖 Loaded {len(df)} rows from the custom dataset")
df.head(3)

📖 Loaded 1500 rows from the custom dataset


,Description,Important Technology Keywords
0,I've been in IT helpdesk for two years and I u...,"Cisco, CCNA, OSPF, BGP, VLANs, ACLs, Network S..."
1,Blockchain technology has fascinated me since ...,"Solidity, Ethereum, EVM, Hardhat, Foundry, Web..."
2,My CS degree touched on machine learning in on...,"Machine Learning, Gradient Descent, Backpropag..."


In [3]:
# Tokenizer + text-to-keywords prompt
PROMPT_PREFIX = "extract important technology keywords: "
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def add_prefix(text: str) -> str:
    return PROMPT_PREFIX + str(text)


def encode_text_pairs(texts, targets):
    input_encodings = tokenizer(
        [add_prefix(text) for text in texts],
        max_length=MAX_INPUT_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="tf",
    )
    target_encodings = tokenizer(
        list(targets),
        max_length=MAX_TARGET_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="tf",
    )
    labels = tf.where(
        target_encodings["input_ids"] == tokenizer.pad_token_id,
        -100,
        target_encodings["input_ids"],
    )
    return input_encodings, tf.cast(labels, tf.int32)

print("✅ Tokenizer ready")

c:\Users\HIMPADMA\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HIMPADMA\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


✅ Tokenizer ready


In [4]:
# 80/20 train-test split and TensorFlow datasets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)

train_inputs, train_labels = encode_text_pairs(
    train_df["Description"].tolist(),
    train_df["Important Technology Keywords"].tolist(),
)
test_inputs, test_labels = encode_text_pairs(
    test_df["Description"].tolist(),
    test_df["Important Technology Keywords"].tolist(),
)

train_ds = tf.data.Dataset.from_tensor_slices(
    (
        {
            "input_ids": train_inputs["input_ids"],
            "attention_mask": train_inputs["attention_mask"],
        },
        train_labels,
    )
).shuffle(1024).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices(
    (
        {
            "input_ids": test_inputs["input_ids"],
            "attention_mask": test_inputs["attention_mask"],
        },
        test_labels,
    )
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"✅ Train rows: {len(train_df)} | Test rows: {len(test_df)}")

✅ Train rows: 1200 | Test rows: 300


In [5]:
# Stronger instruction-tuned seq2seq model
model = TFAutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
model.compile(optimizer=optimizer)

checkpoint_path = os.path.join(CHECKPOINT_DIR, "weights_epoch_{epoch:02d}.weights.h5")
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=True,
    save_best_only=False,
    monitor="val_loss",
    verbose=1,
)
early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
)

print("✅ Model initialized and compiled")

c:\Users\HIMPADMA\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\initializers\initializers.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
All PyTorch model weights were used when initializing TFT5ForConditionalGeneration.

All the weights of TFT5ForConditionalGeneration were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFT5ForConditionalGeneration for predictions without further training.


✅ Model initialized and compiled


In [9]:
# Training cell — safe reinitialization + smoke test
# This cell ensures the runtime uses TensorFlow's Keras (not standalone keras), reinitializes the model,
# and runs a brief smoke training pass.
import sys
import tensorflow as tf
# Force 'keras' imports to reference tf.keras to avoid standalone keras incompatibilities
sys.modules['keras'] = tf.keras
sys.modules['keras.utils'] = tf.keras.utils
sys.modules['keras.losses'] = tf.keras.losses
print(f"✅ sys.modules['keras'] now points to: {sys.modules['keras']} ")

# Recreate and compile the model to ensure it uses the patched keras bindings
from transformers import TFT5ForConditionalGeneration
model = TFT5ForConditionalGeneration.from_pretrained(MODEL_NAME)
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
model.compile(optimizer=optimizer)

# Recreate callbacks (safe to reuse the same paths)
checkpoint_path = os.path.join(CHECKPOINT_DIR, "weights_epoch_{epoch:02d}.weights.h5")
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=True,
    save_best_only=False,
    monitor="val_loss",
    verbose=1,
)
early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
)

# Ensure variables are created via one forward pass with labels
for batch in train_ds.take(1):
    x_sample, y_sample = batch
    _ = model(x_sample, labels=y_sample, training=False)
    break
print("✅ Model reinitialized and variables created")

# Smoke test: 1 epoch on a tiny subset
smoke_train_ds = train_ds.take(1)
smoke_val_ds = val_ds.take(1)

history = model.fit(
    smoke_train_ds,
    validation_data=smoke_val_ds,
    epochs=1,
    callbacks=[checkpoint_callback, early_stopping_callback],
)

print("✅ Smoke training complete")

All PyTorch model weights were used when initializing TFT5ForConditionalGeneration.

All the weights of TFT5ForConditionalGeneration were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFT5ForConditionalGeneration for predictions without further training.


Epoch 1/4
300/300 [==============================] - ETA: 0s - loss: 1.2413
Epoch 1: saving model to c:\Users\HIMPADMA\Desktop\Dokumen\University Work\U'rWay\U'rWay Text Summarization\models\u_rway_keyword_model\checkpoints\weights_epoch_01.weights.h5


c:\Users\HIMPADMA\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\generation\tf_utils.py:465: UserWarning: `seed_generator` is deprecated and will be removed in a future version.
  warnings.warn("`seed_generator` is deprecated and will be removed in a future version.", UserWarning)


300/300 [==============================] - 2119s 7s/step - loss: 1.2413 - val_loss: 0.5141
Epoch 2/4
300/300 [==============================] - ETA: 0s - loss: 0.6049
Epoch 2: saving model to c:\Users\HIMPADMA\Desktop\Dokumen\University Work\U'rWay\U'rWay Text Summarization\models\u_rway_keyword_model\checkpoints\weights_epoch_02.weights.h5
300/300 [==============================] - 1871s 6s/step - loss: 0.6049 - val_loss: 0.2426
Epoch 3/4
300/300 [==============================] - ETA: 0s - loss: 0.3658
Epoch 3: saving model to c:\Users\HIMPADMA\Desktop\Dokumen\University Work\U'rWay\U'rWay Text Summarization\models\u_rway_keyword_model\checkpoints\weights_epoch_03.weights.h5
300/300 [==============================] - 2270s 8s/step - loss: 0.3658 - val_loss: 0.1312
Epoch 4/4
300/300 [==============================] - ETA: 0s - loss: 0.2412
Epoch 4: saving model to c:\Users\HIMPADMA\Desktop\Dokumen\University Work\U'rWay\U'rWay Text Summarization\models\u_rway_keyword_model\checkpoints

,Description,Important Technology Keywords,predicted_keywords
1116,learning pytest properly. write basic tests al...,"Pytest, Fixtures, Parametrize, Mocking, Covera...","Pytest, Fixtures, Parametrize, Mocking, Mockin..."
1368,just want to understand defi well enough to bu...,"Solidity, Hardhat, AMM, Uniswap V2, Constant P...","Solidity, Hardhat, Uniswap V2, Constant Produc..."
422,I write Python scripts at my current job for a...,"AWS, Lambda, Serverless, Terraform, CloudForma...","AWS, Lambda, Serverless, Terraform, CloudForma..."
413,I've been doing manual QA testing at my compan...,"Python, Selenium, Playwright, API Testing, Pos...","Python, Selenium, Playwright, API Testing, Pos..."
451,I've been tinkering with JavaScript for about ...,"JavaScript, Async/Await, React, Redux, Node.js...","JavaScript, Async/Await, React, Node.js, Expre..."
861,i want to build a saas product that ive been t...,"React, FastAPI, Django, PostgreSQL, Stripe, Au...","React, FastAPI, Django, PostgreSQL, Stripe, Au..."
1063,healthcare data analyst is specifically what i...,"SQL, HL7, FHIR, ICD-10, Claims Data, HIPAA, Ta...","SQL, HL7, FHIR, ICD-10, Claims Data, HIPAA, Ta..."
741,react + node + postgres. thats my stack goal. ...,"React, Node.js, PostgreSQL, REST API, JavaScript","React, Node.js, PostgreSQL, REST API, JavaScript"
1272,learning figma from zero. need it for my start...,"Figma, Auto Layout, Components, Variables, Pro...","Figma, Auto Layout, Components, Variables, Pro..."
259,I've been building small interactive web thing...,"JavaScript, Phaser.js, 2D Game Development, En...","JavaScript, Phaser.js, 2D Game Development, En..."


In [10]:
# Final checker: type any new description and get predicted important technology keywords
sample_description = (
    "I want to become a data analyst by learning SQL for joins and window functions, Python with pandas and NumPy for data cleaning, "
    "and Tableau for dashboards. I also want to build Kaggle projects and earn the Google Data Analytics certificate."
)

print("📝 Sample description:")
print(sample_description)
print("\n🧭 Predicted keywords:")
print(predict_keywords(sample_description))

# To test your own description, replace the text below and run this cell again.
your_description = ""
if your_description.strip():
    print("\n✨ Your description prediction:")
    print(predict_keywords(your_description))

📝 Sample description:
I want to become a data analyst by learning SQL for joins and window functions, Python with pandas and NumPy for data cleaning, and Tableau for dashboards. I also want to build Kaggle projects and earn the Google Data Analytics certificate.

🧭 Predicted keywords:
SQL, Joins, Window Functions, Python, Pandas, NumPy, Tableau, Data Cleaning, Google Data Analytics


In [11]:
# Final export for the U'rWay app
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
model.save_pretrained(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

print(f"✅ Saved model and tokenizer to: {MODEL_OUTPUT_DIR}")

# Optional inference example for a roadmap target
sample_paragraph = (
    "I’ve always been interested in how numbers can tell a story about a business, so I want to build a career as a Data Analyst. To get started, I think I need to really sharpen my SQL skills so I can pull data from complex databases without any help. I also want to learn Python, specifically libraries like Pandas and NumPy for cleaning data, and maybe some Scikit-learn because I’m curious about how basic machine learning models work for predicting trends. After I can handle the data, I want to learn a visualization tool like Tableau or Power BI to create dashboards that non-technical people can understand. I’m also planning to work on a few projects using real-world datasets from Kaggle to build a portfolio, and eventually, I’d like to take the Google Data Analytics Professional Certificate to make my resume stand out to recruiters."
)

encoded_sample = tokenizer(
    [add_prefix(sample_paragraph)],
    max_length=MAX_INPUT_LEN,
    padding="max_length",
    truncation=True,
    return_tensors="tf",
)

generated_ids = model.generate(
    input_ids=encoded_sample["input_ids"],
    attention_mask=encoded_sample["attention_mask"],
    max_length=MAX_TARGET_LEN,
)
roadmap_target = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print("🧭 Example roadmap target:", roadmap_target)

✅ Saved model and tokenizer to: c:\Users\HIMPADMA\Desktop\Dokumen\University Work\U'rWay\U'rWay Text Summarization\models\u_rway_keyword_model
🧭 Example roadmap target: SQL, Python, Pandas, NumPy, Scikit-learn, Tableau, Power BI, Real-world Data Analytics, Google Data Analytics


In [12]:
# --- Save / load helper for reuse without retraining ---
# The trained model weights are already saved via `save_pretrained()`.
# This pickle bundle stores the metadata needed to reload the trained model later.
import pickle

PICKLE_PATH = os.path.join(MODEL_OUTPUT_DIR, "u_rway_keyword_bundle.pkl")

bundle = {
    "model_dir": MODEL_OUTPUT_DIR,
    "model_name": MODEL_NAME,
    "prompt_prefix": PROMPT_PREFIX,
    "max_input_len": MAX_INPUT_LEN,
    "max_target_len": MAX_TARGET_LEN,
    "dataset_columns": ["Description", "Important Technology Keywords"],
}

with open(PICKLE_PATH, "wb") as f:
    pickle.dump(bundle, f)

print(f"✅ Saved pickle bundle to: {PICKLE_PATH}")


def load_trained_keyword_model(pickle_path: str = PICKLE_PATH):
    """Reload the trained model/tokenizer without retraining."""
    with open(pickle_path, "rb") as f:
        saved = pickle.load(f)

    loaded_tokenizer = AutoTokenizer.from_pretrained(saved["model_dir"], use_fast=True)
    loaded_model = TFAutoModelForSeq2SeqLM.from_pretrained(saved["model_dir"])
    return loaded_model, loaded_tokenizer, saved


# Example: reload the saved bundle and run one prediction
reloaded_model, reloaded_tokenizer, saved_bundle = load_trained_keyword_model()
print("✅ Reloaded trained model from disk")

example_description = (
    "I want to learn SQL for data analysis, Python pandas for cleaning data, Tableau for dashboards, "
    "and I also want to build Kaggle projects and get the Google Data Analytics certificate."
)

enc = reloaded_tokenizer(
    [saved_bundle["prompt_prefix"] + example_description],
    max_length=saved_bundle["max_input_len"],
    padding="max_length",
    truncation=True,
    return_tensors="tf",
)
ids = reloaded_model.generate(
    input_ids=enc["input_ids"],
    attention_mask=enc["attention_mask"],
    max_length=saved_bundle["max_target_len"],
    num_beams=4,
    early_stopping=True,
)
print("🧭 Reloaded model output:", reloaded_tokenizer.decode(ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=True))

✅ Saved pickle bundle to: c:\Users\HIMPADMA\Desktop\Dokumen\University Work\U'rWay\U'rWay Text Summarization\models\u_rway_keyword_model\u_rway_keyword_bundle.pkl


c:\Users\HIMPADMA\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\initializers\initializers.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
All model checkpoint layers were used when initializing TFT5ForConditionalGeneration.

All the layers of TFT5ForConditionalGeneration were initialized from the model checkpoint at c:\Users\HIMPADMA\Desktop\Dokumen\University Work\U'rWay\U'rWay Text Summarization\models\u_rway_keyword_model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFT5ForConditionalGeneration for predictions without further training.


✅ Reloaded trained model from disk
🧭 Reloaded model output: SQL, Python, Pandas, Tableau, Kaggle, Data Analytics
